# Statistics Villains: Web Scraping with pandas
## MAD 2502 — Applied Data

Use the class Statistics Villains page to practice this workflow:

**webpage → HTML → table → DataFrame → cleaning → analysis**

For your own projects, prefer an API or downloadable file when one exists, and always check a site's access rules.


In [ ]:
import pandas as pd
import requests

PAGE_URL = (
    "https://people.clas.ufl.edu/mathguy/"
    "supervillain-statistics-playground/"
)


## 1. Look before you scrape

A common first check is the site's `robots.txt` file. For this class example, the Statistics Villains page was created for teaching, so we will make only a small number of requests.


In [ ]:
robots_url = "https://people.clas.ufl.edu/robots.txt"

robots_response = requests.get(robots_url, timeout=30)
print("robots.txt status:", robots_response.status_code)
print(robots_response.text[:800])


## 2. Download the page

`requests.get()` retrieves the page. `raise_for_status()` stops the program if the request was unsuccessful.


In [ ]:
response = requests.get(PAGE_URL, timeout=30)
response.raise_for_status()

html = response.text

print("status:", response.status_code)
print("characters:", len(html))
print(html[:400])


## 3. Find HTML tables

`pd.read_html()` returns a **list of DataFrames**, because one webpage can contain several tables.


In [ ]:
tables = pd.read_html(html)

print("tables found:", len(tables))
for index, table in enumerate(tables):
    print(index, table.shape, list(table.columns))


In [ ]:
villains = tables[0].copy()

print("shape:", villains.shape)
print("columns:", list(villains.columns))
print(villains.dtypes)
display(villains.head())


## 4. Clean column names

Web tables are written for people, not Python. Simple column names are easier to type and debug.


In [ ]:
villains.columns = [
    str(name)
    .strip()
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
    for name in villains.columns
]

print(list(villains.columns))
display(villains.head())


## 5. Clean a money column

First inspect the column names. If one contains `insurance` or `cost`, remove `$` and commas before converting it to numbers.


In [ ]:
money_columns = [
    name
    for name in villains.columns
    if "insurance" in name or "cost" in name
]

print("possible money columns:", money_columns)

if money_columns:
    money = money_columns[0]

    cleaned = (
        villains[money]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    villains[money] = pd.to_numeric(
        cleaned,
        errors="coerce",
    )

    display(villains[[money]].head())


## 6. Ask questions

Try these patterns after looking at the actual columns in your table.


In [ ]:
print("numeric summaries:")
display(villains.describe(numeric_only=True))

print("\nfirst five rows:")
display(villains.head())


In [ ]:
if money_columns:
    money = money_columns[0]

    print("mean:", villains[money].mean())
    print("median:", villains[money].median())

    display(
        villains.sort_values(
            money,
            ascending=False,
        ).head(5)
    )


## 7. Save a cleaned copy


In [ ]:
villains.to_csv(
    "statistics_villains_clean.csv",
    index=False,
)

print("saved statistics_villains_clean.csv")


# Practice

1. What does one row represent?
2. How many rows and columns were retrieved?
3. Choose one numeric column and compare its mean and median.
4. Find the five largest values in a numeric column.
5. Use `value_counts()` on a categorical column.
6. Explain why blindly using `tables[0]` could fail on another website.
7. Explain why an API is usually preferable when a site provides one.
8. Add one labeled graph that answers a question about the villains.

### Capstone connection

Before scraping a new site, ask:

- Is there an API or CSV instead?
- What does one row represent?
- Is automated access allowed?
- Can the page structure change?
- Are you collecting only what you need?
